In [2]:
from pyspark.sql import SparkSession
from pyspark import SparkContext

SparkContext.getOrCreate()

spark = SparkSession\
        .builder\
        .master("local")\
        .appName("Introduction au DataFrame")\
        .getOrCreate()

spark


your 131072x1 screen size is bogus. expect trouble
24/01/03 20:53:30 WARN Utils: Your hostname, JayWin10Pc resolves to a loopback address: 127.0.1.1; using 172.17.52.247 instead (on interface eth0)
24/01/03 20:53:30 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/01/03 20:53:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
import os
pwd = os.getcwd()
print(pwd)
print("{}/data/miserables_full.txt".format(pwd))

/home/jayl/datascientestfile/spark
/home/jayl/datascientestfile/spark/data/miserables_full.txt


In [4]:
sc = SparkContext.getOrCreate()
sc


<SparkContext master=local[*] appName=pyspark-shell>

In [5]:
from pyspark.sql import Row 
rdd = sc.textFile("{}/data/2008.csv".format(pwd)).map(lambda x:x.split(","))
rdd_row = rdd.map(lambda x:Row(annee = x[0],mois =x[1],jours =x[2],flightNum = x[9]))

df = spark.createDataFrame(rdd_row)

In [6]:
# df.head(5)
df.show(5)
# rdd_row.take()

+-------+------+-------+-----------+
|  annee|  mois|  jours|  flightNum|
+-------+------+-------+-----------+
|"annee"|"mois"|"jours"|"flightNum"|
|   2008|     1|      1|        324|
|   2008|     1|      1|        572|
|   2008|     1|      1|        511|
|   2008|     1|      1|        376|
+-------+------+-------+-----------+
only showing top 5 rows



In [7]:
raw_df = spark.read.csv("{}/data/2008.csv".format(pwd),header = True)
raw_df.printSchema()

root
 |-- annee: string (nullable = true)
 |-- mois: string (nullable = true)
 |-- jours: string (nullable = true)
 |-- heure: string (nullable = true)
 |-- _c4: string (nullable = true)
 |-- _c5: string (nullable = true)
 |-- _c6: string (nullable = true)
 |-- _c7: string (nullable = true)
 |-- uniqueCarrier: string (nullable = true)
 |-- flightNum: string (nullable = true)
 |-- tailNum: string (nullable = true)
 |-- _c11: string (nullable = true)
 |-- _c12: string (nullable = true)
 |-- _c13: string (nullable = true)
 |-- _c14: string (nullable = true)
 |-- _c15: string (nullable = true)
 |-- origin: string (nullable = true)
 |-- dest: string (nullable = true)
 |-- distance: string (nullable = true)
 |-- canceled: string (nullable = true)
 |-- cancellationCode: string (nullable = true)
 |-- diverted: string (nullable = true)
 |-- carrierDelay: string (nullable = true)
 |-- weatherDelay: string (nullable = true)
 |-- nasDelay: string (nullable = true)
 |-- securityDelay: string (nulla

In [8]:
flights1 = raw_df.select('annee', 'mois', 'jours', 'flightNum', 'origin', 'dest', 'distance', 'canceled', 'cancellationCode', 'carrierDelay')

type(flights1)

flights1.show()

+-----+----+-----+---------+------+----+--------+--------+----------------+------------+
|annee|mois|jours|flightNum|origin|dest|distance|canceled|cancellationCode|carrierDelay|
+-----+----+-----+---------+------+----+--------+--------+----------------+------------+
| 2008|   1|    1|      324|   SEA| SJC|     697|       7|              16|        NULL|
| 2008|   1|    1|      572|   SEA| PSP|     987|       6|              25|        NULL|
| 2008|   1|    1|      511|   SAN| SEA|    1050|       7|              14|        NULL|
| 2008|   1|    1|      376|   SEA| GEG|     224|       5|              13|        NULL|
| 2008|   1|    1|      729|   TUS| SEA|    1216|       6|              11|        NULL|
| 2008|   1|    1|      283|   LAX| SEA|     954|       7|              16|        NULL|
| 2008|   1|    1|      211|   LAX| SEA|     954|       6|              17|        NULL|
| 2008|   1|    1|      100|   ANC| PDX|    1542|       2|              14|        NULL|
| 2008|   1|    1|   

In [9]:
# Création d'un DataFrame en spécifiant le type des colonnes
flights = raw_df.select(raw_df.annee.cast("int"),
                        raw_df.mois.cast("int"),
                        raw_df.jours.cast("int"),
                        raw_df.flightNum.cast("int"),
                        raw_df.origin.cast("string"),
                        raw_df.dest.cast("string"),
                        raw_df.distance.cast("int"),
                        raw_df.canceled.cast("boolean"),
                        raw_df.cancellationCode.cast("string"),
                        raw_df.carrierDelay.cast("int"))

# Affichage de 20 premières lignes
flights.show()

+-----+----+-----+---------+------+----+--------+--------+----------------+------------+
|annee|mois|jours|flightNum|origin|dest|distance|canceled|cancellationCode|carrierDelay|
+-----+----+-----+---------+------+----+--------+--------+----------------+------------+
| 2008|   1|    1|      324|   SEA| SJC|     697|    NULL|              16|        NULL|
| 2008|   1|    1|      572|   SEA| PSP|     987|    NULL|              25|        NULL|
| 2008|   1|    1|      511|   SAN| SEA|    1050|    NULL|              14|        NULL|
| 2008|   1|    1|      376|   SEA| GEG|     224|    NULL|              13|        NULL|
| 2008|   1|    1|      729|   TUS| SEA|    1216|    NULL|              11|        NULL|
| 2008|   1|    1|      283|   LAX| SEA|     954|    NULL|              16|        NULL|
| 2008|   1|    1|      211|   LAX| SEA|     954|    NULL|              17|        NULL|
| 2008|   1|    1|      100|   ANC| PDX|    1542|    NULL|              14|        NULL|
| 2008|   1|    1|   

In [10]:
raw_df.select("flightNum").distinct().count()

379

In [13]:
flights.describe().show(truncate=8)

+-------+------+----+--------+---------+------+----+--------+----------------+------------+
|summary| annee|mois|   jours|flightNum|origin|dest|distance|cancellationCode|carrierDelay|
+-------+------+----+--------+---------+------+----+--------+----------------+------------+
|  count|  9999|9999|    9999|     9999|  9999|9999|    9999|            9999|           0|
|   mean|2008.0| 1.0|12.59...| 354.3...|  NULL|NULL|920.5...|        16.07...|        NULL|
| stddev|   0.0| 0.0|6.996...| 231.8...|  NULL|NULL|587.6...|        7.542...|        NULL|
|    min|  2008|   1|       1|        1|   ADK| ADK|      31|              10|        NULL|
|    max|  2008|   1|      25|      954|   YAK| YAK|    2846|              NA|        NULL|
+-------+------+----+--------+---------+------+----+--------+----------------+------------+



In [12]:

flights.describe().toPandas()

ImportError: Pandas >= 1.0.5 must be installed; however, it was not found.

In [ ]:

import pandas as pd

arr = pd.array([1,2])
print(arr[0])

ModuleNotFoundError: No module named 'pandas'

ModuleNotFoundError: No module named 'pandas'

In [14]:
flights.groupBy("cancellationCode").count().show()

+----------------+-----+
|cancellationCode|count|
+----------------+-----+
|               7|  161|
|              51|    7|
|              15|  667|
|              54|    3|
|              11|  681|
|              29|   61|
|              69|    2|
|              42|   10|
|               3|   49|
|              30|   67|
|             113|    1|
|              34|   31|
|              59|    2|
|               8|  283|
|              22|  249|
|              28|   99|
|              16|  588|
|              35|   18|
|              52|    4|
|              NA|  269|
+----------------+-----+
only showing top 20 rows



In [15]:
flights.groupBy("cancellationCode","canceled").count().show()

+----------------+--------+-----+
|cancellationCode|canceled|count|
+----------------+--------+-----+
|              69|    true|    1|
|               4|    true|    1|
|               7|    NULL|  157|
|              51|    NULL|    7|
|              15|    NULL|  664|
|              54|    NULL|    3|
|              11|    NULL|  677|
|              30|    true|    1|
|               3|    true|    4|
|              29|    NULL|   61|
|              69|    NULL|    1|
|              42|    NULL|   10|
|               3|    NULL|   45|
|              30|    NULL|   66|
|             113|    NULL|    1|
|              34|    NULL|   31|
|              59|    NULL|    2|
|               8|    NULL|  283|
|              13|    true|    2|
|              22|    NULL|  247|
+----------------+--------+-----+
only showing top 20 rows



In [19]:
flights.filter(flights.cancellationCode == "C").show()

+-----+----+-----+---------+------+----+--------+--------+----------------+------------+
|annee|mois|jours|flightNum|origin|dest|distance|canceled|cancellationCode|carrierDelay|
+-----+----+-----+---------+------+----+--------+--------+----------------+------------+
+-----+----+-----+---------+------+----+--------+--------+----------------+------------+



In [22]:
# 因为截取10000行，只有一月数据，所以换成日
flights.filter(flights.canceled == True).groupBy("jours").count().show()

+-----+-----+
|jours|count|
+-----+-----+
|   12|    1|
|    1|    1|
|   13|    5|
|    6|    2|
|   16|    4|
|    3|    3|
|   20|    3|
|   19|    2|
|   15|    1|
|    9|    1|
|   17|    2|
|    4|    2|
|    8|    1|
|   23|    1|
|    7|    1|
|   25|    1|
|   24|    1|
|   21|    3|
|   11|    3|
|   14|    2|
+-----+-----+
only showing top 20 rows



In [25]:
flights.withColumn("isLongFlight",flights.distance > 1000).show(10)
# 注意这里是创造了一个新的dataFrame，原df是不变的
# flights.show(10)

+-----+----+-----+---------+------+----+--------+--------+----------------+------------+------------+
|annee|mois|jours|flightNum|origin|dest|distance|canceled|cancellationCode|carrierDelay|isLongFlight|
+-----+----+-----+---------+------+----+--------+--------+----------------+------------+------------+
| 2008|   1|    1|      324|   SEA| SJC|     697|    NULL|              16|        NULL|       false|
| 2008|   1|    1|      572|   SEA| PSP|     987|    NULL|              25|        NULL|       false|
| 2008|   1|    1|      511|   SAN| SEA|    1050|    NULL|              14|        NULL|        true|
| 2008|   1|    1|      376|   SEA| GEG|     224|    NULL|              13|        NULL|       false|
| 2008|   1|    1|      729|   TUS| SEA|    1216|    NULL|              11|        NULL|        true|
| 2008|   1|    1|      283|   LAX| SEA|     954|    NULL|              16|        NULL|       false|
| 2008|   1|    1|      211|   LAX| SEA|     954|    NULL|              17|       

In [32]:
flights.fillna(0,"carrierDelay").show(6)
# 注意这里也是不会修改原df的
flights.show(6)

+-----+----+-----+---------+------+----+--------+--------+----------------+------------+
|annee|mois|jours|flightNum|origin|dest|distance|canceled|cancellationCode|carrierDelay|
+-----+----+-----+---------+------+----+--------+--------+----------------+------------+
| 2008|   1|    1|      324|   SEA| SJC|     697|    NULL|              16|           0|
| 2008|   1|    1|      572|   SEA| PSP|     987|    NULL|              25|           0|
| 2008|   1|    1|      511|   SAN| SEA|    1050|    NULL|              14|           0|
| 2008|   1|    1|      376|   SEA| GEG|     224|    NULL|              13|           0|
| 2008|   1|    1|      729|   TUS| SEA|    1216|    NULL|              11|           0|
| 2008|   1|    1|      283|   LAX| SEA|     954|    NULL|              16|           0|
+-----+----+-----+---------+------+----+--------+--------+----------------+------------+
only showing top 6 rows

+-----+----+-----+---------+------+----+--------+--------+----------------+----------

In [34]:
flights.replace(['A','B','C'],['1', '2','3'],"cancellationCode").show(10)


+-----+----+-----+---------+------+----+--------+--------+----------------+------------+
|annee|mois|jours|flightNum|origin|dest|distance|canceled|cancellationCode|carrierDelay|
+-----+----+-----+---------+------+----+--------+--------+----------------+------------+
| 2008|   1|    1|      324|   SEA| SJC|     697|    NULL|              16|        NULL|
| 2008|   1|    1|      572|   SEA| PSP|     987|    NULL|              25|        NULL|
| 2008|   1|    1|      511|   SAN| SEA|    1050|    NULL|              14|        NULL|
| 2008|   1|    1|      376|   SEA| GEG|     224|    NULL|              13|        NULL|
| 2008|   1|    1|      729|   TUS| SEA|    1216|    NULL|              11|        NULL|
| 2008|   1|    1|      283|   LAX| SEA|     954|    NULL|              16|        NULL|
| 2008|   1|    1|      211|   LAX| SEA|     954|    NULL|              17|        NULL|
| 2008|   1|    1|      100|   ANC| PDX|    1542|    NULL|              14|        NULL|
| 2008|   1|    1|   